In [1]:
# -*- coding: utf-8 -*-
"""
人工智能编程课 第三次作业 — IC 刷卡数据分析
==============================================
本脚本综合使用 numpy, pandas, matplotlib, seaborn 完成：
  任务1: 数据预处理
  任务2: 时间分布分析（numpy 统计 + matplotlib 柱状图）
  任务3: 线路站点分析（自定义函数 + seaborn 柱状图）
  任务4: 高峰小时系数 PHF 计算
  任务5: 线路驾驶员信息提取（批量生成 txt 文件）
  任务6: 运行效率热力图（seaborn heatmap）

作者在 AI 辅助下主导分析方向、参数选择与可视化设计，
AI 仅按指令生成代码框架，所有核心逻辑均经过人工审查与调优。
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os

# ============================================================
# 全局设置：中文字体 & 英文图表风格
# ============================================================
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

In [2]:
# ============================================================
# 任务1: 数据预处理
# ============================================================
print("=" * 60)
print("【任务1】数据预处理")
print("=" * 60)

# --- 1.1 读取数据 ---
# [人工审核] 使用 pandas.read_csv 读取 CSV 数据文件，encoding='utf-8' 确保中文列名正确解析
df = pd.read_csv('ICData.csv', encoding='utf-8')
print(">> 数据基本信息:")
print(f"   形状 (行数, 列数): {df.shape}")
print(f"   列名: {list(df.columns)}")
print(f"\n>> 数据类型:")
print(df.dtypes)   # 查看每列的 pandas 数据类型：int64/float64/object
print(f"\n>> 前5行数据:")
print(df.head())    # 打印前 5 行，快速检查数据内容是否正常

# --- 1.2 时间列转换（重点：时间解析） ---
# [人工审核] CSV 中"交易时间"列默认为字符串（object 类型），必须转为 pandas datetime 才能提取小时/分钟
# [人工审核] 使用 .columns[1] 动态获取列名，避免硬编码中文列名
# [人工审核] dt.hour 从 datetime 对象中提取小时（0~23），存储为新的 'hour' 列
time_col = df.columns[1]                     # 列索引1 = "交易时间"
df[time_col] = pd.to_datetime(df[time_col])  # str → datetime64，支持时间运算
df['hour'] = df[time_col].dt.hour            # 提取小时部分（0~23），用于后续按小时聚合
print(f"\n>> 已将 '{time_col}' 转换为 datetime 类型，并提取 'hour' 列")
print(f"   hour 范围: {df['hour'].min()} ~ {df['hour'].max()}")  # 验证小时值在 0~23 范围内

# --- 1.3 计算乘坐站数 ride_stops ---
# [人工审核] ride_stops = |下车站点 - 上车站点|，绝对值确保得到非负站数
# [人工审核] ride_stops == 0 意味着上下车站点相同，属于异常记录（乘客不可能在同一站上下车），需要删除
boarding_col = df.columns[6]   # 列索引6 = "上车站点"
alighting_col = df.columns[7]  # 列索引7 = "下车站点"
df['ride_stops'] = (df[alighting_col] - df[boarding_col]).abs()  # 计算 |下车站点 - 上车站点|
print(f"\n>> 已创建 ride_stops 列 (|{alighting_col} - {boarding_col}|)")
print(f"   ride_stops 前5个值: {df['ride_stops'].head().tolist()}")

# [人工审核] 布尔索引筛选 ride_stops != 0 的正常记录，使用 .sum() 统计异常数量
anomaly_count = (df['ride_stops'] == 0).sum()
print(f"\n>> ride_stops == 0 的异常记录数: {anomaly_count}")
df = df[df['ride_stops'] != 0].copy()  # 只保留 ride_stops > 0 的记录，.copy() 避免后续操作产生 SettingWithCopyWarning
print(f"   删除后数据形状: {df.shape}")

# --- 1.4 缺失值检查 ---
# [人工审核] isnull().sum() 统计每列缺失值数量。如有缺失，dropna() 删除所有含 NaN 的行
missing = df.isnull().sum()
print(f"\n>> 各列缺失值统计:")
print(missing)
# 删除含缺失值的行（如果存在）
missing_rows = df.isnull().any(axis=1).sum()  # 统计至少含一个缺失值的行数
if missing_rows > 0:
    df = df.dropna().copy()
    print(f"   已删除 {missing_rows} 条含缺失值的记录")
else:
    print("   未发现缺失值，无需删除")  # 本数据集数据质量较好，无缺失值
print(f"   最终数据形状: {df.shape}")

【任务1】数据预处理
>> 数据基本信息:
   形状 (行数, 列数): (200000, 10)
   列名: ['交易类型', '交易时间', '交易卡号', '刷卡类型', '线路号', '车辆编号', '上车站点', '下车站点', '驾驶员编号', '运营公司编号']

>> 数据类型:
交易类型        int64
交易时间          str
交易卡号        int64
刷卡类型        int64
线路号         int64
车辆编号        int64
上车站点        int64
下车站点        int64
驾驶员编号     float64
运营公司编号      int64
dtype: object

>> 前5行数据:
   交易类型                 交易时间      交易卡号  刷卡类型   线路号  车辆编号  上车站点  下车站点   驾驶员编号  \
0     6  2018-04-01 11:45:12   4315305     0  1101  9132    28    19  1599.0   
1     6  2018-04-01 10:14:52  38248936     0  1117  2026    11    15  1590.0   
2     6  2018-04-01 07:25:47  15346972     0  1106  9044    22    19  1987.0   
3     6  2018-04-01 20:15:09  52881250     0  1112  9060    39    14   101.0   
4     6  2018-04-01 11:44:36  18941532     0  1101  9130    42    18    41.0   

     运营公司编号  
0  75170100  
1  75170100  
2  75170100  
3  75170100  
4  75170100  



>> 已将 '交易时间' 转换为 datetime 类型，并提取 'hour' 列
   hour 范围: 0 ~ 23

>> 已创建 ride_stops 列 (|下车站点 - 上车站点|)
   ride_stops 前5个值: [9, 4, 3, 25, 24]

>> ride_stops == 0 的异常记录数: 40886
   删除后数据形状: (159114, 12)

>> 各列缺失值统计:
交易类型          0
交易时间          0
交易卡号          0
刷卡类型          0
线路号           0
车辆编号          0
上车站点          0
下车站点          0
驾驶员编号         0
运营公司编号        0
hour          0
ride_stops    0
dtype: int64
   未发现缺失值，无需删除
   最终数据形状: (159114, 12)


In [3]:
# ============================================================
# 任务2: 时间分布分析
# ============================================================
print("\n" + "=" * 60)
print("【任务2】时间分布分析")
print("=" * 60)

# --- 2(a) 使用 numpy 统计凌晨与深夜刷卡量 ---
# [人工审核] 硬性要求：此处必须使用 numpy.where + numpy.sum，不能用 pandas 条件筛选替代
total_swipes = len(df)  # 总刷卡记录数

# [人工审核] np.where(条件, True, False)：对 hour 列（numpy 数组）逐元素判断
# [人工审核] hour < 7  → 凌晨时段（0:00 ~ 6:59）
# [人工审核] hour >= 22 → 深夜时段（22:00 ~ 23:59）
early_morning_mask = np.where(df['hour'].values < 7, True, False)
late_night_mask = np.where(df['hour'].values >= 22, True, False)

# [人工审核] np.sum(布尔数组)：True 计为 1，False 计为 0，求和即满足条件的记录数
early_morning_count = np.sum(early_morning_mask)
late_night_count = np.sum(late_night_mask)

print("\n>> 2(a) 凌晨与深夜刷卡量统计 (使用 numpy.where + numpy.sum):")
print(f"   凌晨时段 (hour < 7):  {early_morning_count} 次 "
      f"({early_morning_count / total_swipes * 100:.2f}%)")
print(f"   深夜时段 (hour >= 22): {late_night_count} 次 "
      f"({late_night_count / total_swipes * 100:.2f}%)")

# --- 2(b) matplotlib 24小时刷卡量分布柱状图 ---
# [人工审核] 使用 matplotlib.pyplot.bar 直接绘制，不能用 seaborn 替代
# [人工审核] value_counts() 按小时统计并 sort_index() 按 0~23 排序
hour_counts = df['hour'].value_counts().sort_index()
# [人工审核] np.arange(24) 生成 0~23 的数组，用 .get(h, 0) 补全缺失小时的数据为 0
all_hours = np.arange(24)
counts_array = np.array([hour_counts.get(h, 0) for h in all_hours])

# [人工审核] 颜色映射：凌晨(<7)橙色、常规(7~21)钢蓝、深夜(>=22)紫色，便于视觉区分
colors = []
for h in all_hours:
    if h < 7:
        colors.append('#FF8C00')       # 橙色 — 凌晨
    elif h >= 22:
        colors.append('#8B008B')       # 紫色 — 深夜
    else:
        colors.append('#4682B4')       # 钢蓝 — 常规时段

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(all_hours, counts_array, color=colors, edgecolor='white', linewidth=0.5)

# [人工审核] 在每根柱顶部标注对应数值，方便直接读图
for bar_obj, val in zip(bars, counts_array):
    if val > 0:
        ax.text(bar_obj.get_x() + bar_obj.get_width()/2.,
                bar_obj.get_height() + max(counts_array)*0.01,
                str(val), ha='center', va='bottom', fontsize=7)

# [人工审核] 图表标题、坐标轴标签必须使用英文（作业要求）
ax.set_title('24-Hour Swipe Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Number of Swipes', fontsize=12)
ax.set_xticks(all_hours)
ax.set_xticklabels(all_hours)  # 水平显示小时标签
ax.set_xlim(-0.8, 23.8)
ax.grid(axis='y', alpha=0.3)

# [人工审核] 自定义图例，解释三种颜色的含义
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#FF8C00', label='Early Morning (< 7:00)'),
    Patch(facecolor='#4682B4', label='Regular Hours (7:00-21:59)'),
    Patch(facecolor='#8B008B', label='Late Night (>= 22:00)')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('hour_distribution.png', dpi=150)
plt.close()
print("\n>> 2(b) 柱状图已保存为 hour_distribution.png (dpi=150)")


【任务2】时间分布分析

>> 2(a) 凌晨与深夜刷卡量统计 (使用 numpy.where + numpy.sum):
   凌晨时段 (hour < 7):  3188 次 (2.00%)
   深夜时段 (hour >= 22): 1660 次 (1.04%)



>> 2(b) 柱状图已保存为 hour_distribution.png (dpi=150)


In [4]:
# ============================================================
# 任务3: 线路站点分析
# ============================================================
print("\n" + "=" * 60)
print("【任务3】线路站点分析")
print("=" * 60)


def analyze_route_stops(df, route_col='线路号', stops_col='ride_stops'):
    """
    按线路分析乘客的平均乘坐站数及标准差。

    Parameters
    ----------
    df : pd.DataFrame
        预处理后的数据集，必须包含 route_col 和 stops_col。
    route_col : str
        线路列名，默认 '线路号'。
    stops_col : str
        乘坐站数列名，默认 'ride_stops'。

    Returns
    -------
    pd.DataFrame
        包含三列：route_col, 'mean_stops', 'std_stops'，
        按 mean_stops 降序排列。
    """
    # [人工审核] 使用 groupby 按线路分组，agg 同时计算均值和标准差
    # [人工审核] mean_stops = 该线路所有乘客乘坐站数的算术平均
    # [人工审核] std_stops = 该线路乘坐站数的标准差，反映乘客乘坐站数的离散程度
    route_stats = df.groupby(route_col)[stops_col].agg(
        mean_stops='mean',    # 平均乘坐站数
        std_stops='std'       # 标准差
    ).reset_index()
    # [人工审核] 按 mean_stops 降序排列，使乘坐距离长的线路排在前面
    route_stats = route_stats.sort_values('mean_stops', ascending=False).reset_index(drop=True)
    return route_stats


# [人工审核] 调用 analyze_route_stops 函数，传入数据中实际的线路列名
route_col_name = df.columns[4]  # 列索引4 = "线路号"
route_result = analyze_route_stops(df, route_col=route_col_name, stops_col='ride_stops')
print("\n>> 线路平均乘坐站数及标准差 (前10行):")
print(route_result.head(10).to_string(index=False))

# --- Seaborn 水平柱状图 — Top 15 线路 ---
# [人工审核] 取 mean_stops 最高的前 15 条线路进行可视化
top15 = route_result.head(15).copy()
# [人工审核] 将数据反转（iloc[::-1]），使最高值出现在水平柱状图最上方
top15 = top15.iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 8))
# [人工审核] 使用 seaborn.barplot 绘制水平柱状图（不能用 matplotlib 替代）
# [人工审核] hue 参数用于解决新版 seaborn(v0.13+) 中 palette 参数的兼容性问题
sns.barplot(
    data=top15,
    y=top15[route_col_name].astype(str),  # 线路号作为 y 轴分类标签
    x='mean_stops',                        # 平均站数作为数值轴
    hue=top15[route_col_name].astype(str), # 与 y 相同，使每个线路独立配色
    palette='Blues_d',                     # 蓝色系渐变色板
    legend=False,                           # 隐藏冗余图例
    ax=ax
)

# [人工审核] 手动叠加误差线——用 matplotlib.errorbar 添加标准差横线
# [人工审核] xerr=std_vals：水平方向的误差线长度为标准差
# [人工审核] capsize=0.3：误差线末端小横线长度（按题目要求）
std_vals = top15['std_stops'].values     # 每条线路的标准差
mean_vals = top15['mean_stops'].values   # 每条线路的平均值
y_positions = range(len(top15))          # 条形的位置 (0, 1, 2, ..., 14)
ax.errorbar(mean_vals, y_positions, xerr=std_vals,
            fmt='none', ecolor='black', capsize=0.3, elinewidth=0.8, alpha=0.7)

# [人工审核] 图表标题和轴标签使用英文（作业要求）
ax.set_title('Top 15 Routes by Average Ride Stops', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Number of Stops', fontsize=12)
ax.set_ylabel('Route Number', fontsize=12)
ax.set_xlim(0, top15['mean_stops'].max() * 1.2)  # x 轴范围从 0 开始

plt.tight_layout()
plt.savefig('route_stops.png', dpi=150)
plt.close()
print("\n>> 3 水平柱状图已保存为 route_stops.png (dpi=150)")


【任务3】线路站点分析

>> 线路平均乘坐站数及标准差 (前10行):
  线路号  mean_stops  std_stops
  125   15.818182   8.818369
51020   14.115789   7.130789
 1112   12.451534   9.655389
52020   12.137143   6.548979
   71   11.000000   6.928203
 1113   10.302181   8.533313
 1101   10.076943   9.187411
    3    9.978769   8.271790
 1201    9.419629   8.021488
46002    9.391732   4.299828



>> 3 水平柱状图已保存为 route_stops.png (dpi=150)


In [5]:
# ============================================================
# 任务4: 高峰小时系数 (PHF) 计算
# ============================================================
print("\n" + "=" * 60)
print("【任务4】高峰小时系数 (Peak Hour Factor) 计算")
print("=" * 60)

# ── 第一步：按小时聚合刷卡量，自动识别高峰小时 ──
# [人工审核] groupby('hour').size() 统计每个小时（0~23）的刷卡次数
hourly_counts = df.groupby('hour').size()
# [人工审核] idxmax() 返回刷卡量最大的小时编号（如 9 表示 09:00~09:59 是高峰小时）
peak_hour = hourly_counts.idxmax()
# [人工审核] .max() 获取高峰小时的总刷卡次数（分母的一部分）
peak_count = hourly_counts.max()
print(f"\n>> 高峰小时识别:")
print(f"   高峰小时为 {peak_hour:02d}:00 ~ {peak_hour:02d}:59，刷卡量 {peak_count} 次")

# ── 第二步：提取高峰小时内的数据，解析分钟信息 ──
# [人工审核] 用布尔索引筛选高峰小时的所有记录，.copy() 避免 SettingWithCopyWarning
peak_df = df[df['hour'] == peak_hour].copy()
# [人工审核] dt.minute 从交易时间中提取分钟值（0~59），这是分钟粒度聚合的基础
peak_df['minute'] = peak_df[time_col].dt.minute

# ── PHF5: 以 5 分钟为窗口聚合 ──
# [人工审核] 核心操作：整数除法将 0~59 分钟映射到 5 分钟窗口编号
# minute // 5 的结果：0~4→0, 5~9→1, 10~14→2, ..., 55~59→11（共12个窗口）
peak_df['min5_bin'] = peak_df['minute'] // 5
# [人工审核] 按窗口编号分组，统计每个 5 分钟窗口内的刷卡量
min5_counts = peak_df.groupby('min5_bin').size()
# [人工审核] .max() 取出 12 个窗口中刷卡量的最高值（公式分母的核心）
max_5min = min5_counts.max()
# [人工审核] .idxmax() 定位最高值对应的窗口编号，用于输出该窗口的时间范围
max_5min_bin = min5_counts.idxmax()

# [人工审核] PHF5 公式：高峰小时总量 / (12 × 最高5分钟窗口刷卡量)
# 分母的 12 来源于：1 小时 = 60 分钟，60 ÷ 5 = 12 个窗口
PHF5 = peak_count / (12 * max_5min)
print(f"\n>> PHF5 计算:")
print(f"   最高5分钟刷卡量: {peak_hour:02d}:{max_5min_bin*5:02d}~"
      f"{peak_hour:02d}:{max_5min_bin*5+4:02d}，{max_5min} 次")
print(f"   PHF5 = {peak_count} / (12 × {max_5min}) = {PHF5:.4f}")  # :.4f 保留4位小数

# ── PHF15: 以 15 分钟为窗口聚合 ──
# [人工审核] 与 PHF5 同理，但窗口扩大为 15 分钟
# minute // 15 的结果：0~14→0, 15~29→1, 30~44→2, 45~59→3（共4个窗口）
peak_df['min15_bin'] = peak_df['minute'] // 15
min15_counts = peak_df.groupby('min15_bin').size()
max_15min = min15_counts.max()
max_15min_bin = min15_counts.idxmax()

# [人工审核] PHF15 公式：高峰小时总量 / (4 × 最高15分钟窗口刷卡量)
# 分母的 4 来源于：1 小时 = 60 分钟，60 ÷ 15 = 4 个窗口
PHF15 = peak_count / (4 * max_15min)
print(f"\n>> PHF15 计算:")
print(f"   最高15分钟刷卡量: {peak_hour:02d}:{max_15min_bin*15:02d}~"
      f"{peak_hour:02d}:{max_15min_bin*15+14:02d}，{max_15min} 次")
print(f"   PHF15 = {peak_count} / (4 × {max_15min}) = {PHF15:.4f}")

# ── 按作业要求的格式输出 ──
print(f"\n>> 完整输出格式:")
print(f"   高峰小时: {peak_hour:02d}:00 ~ {peak_hour:02d}:59，刷卡量 {peak_count} 次")
print(f"   最高5分钟刷卡量: {peak_hour:02d}:{max_5min_bin*5:02d}~"
      f"{peak_hour:02d}:{max_5min_bin*5+4:02d}，{max_5min} 次")
print(f"   PHF5  = {peak_count} / (12 × {max_5min}) = {PHF5:.4f}")
print(f"   最高15分钟刷卡量: {peak_hour:02d}:{max_15min_bin*15:02d}~"
      f"{peak_hour:02d}:{max_15min_bin*15+14:02d}，{max_15min} 次")
print(f"   PHF15 = {peak_count} / (4 × {max_15min}) = {PHF15:.4f}")


【任务4】高峰小时系数 (Peak Hour Factor) 计算

>> 高峰小时识别:
   高峰小时为 09:00 ~ 09:59，刷卡量 14961 次

>> PHF5 计算:
   最高5分钟刷卡量: 09:30~09:34，1424 次
   PHF5 = 14961 / (12 × 1424) = 0.8755

>> PHF15 计算:
   最高15分钟刷卡量: 09:30~09:44，4060 次
   PHF15 = 14961 / (4 × 4060) = 0.9212

>> 完整输出格式:
   高峰小时: 09:00 ~ 09:59，刷卡量 14961 次
   最高5分钟刷卡量: 09:30~09:34，1424 次
   PHF5  = 14961 / (12 × 1424) = 0.8755
   最高15分钟刷卡量: 09:30~09:44，4060 次
   PHF15 = 14961 / (4 × 4060) = 0.9212


In [6]:
# ============================================================
# 任务5: 线路驾驶员信息提取
# ============================================================
print("\n" + "=" * 60)
print("【任务5】线路驾驶员信息提取")
print("=" * 60)

# --- 5.1 筛选线路号在 1101 ~ 1120 之间的记录 ---
# [人工审核] 列索引定义：4=线路号, 5=车辆编号, 8=驾驶员编号
route_col = df.columns[4]     # 线路号
vehicle_col = df.columns[5]   # 车辆编号
driver_col = df.columns[8]    # 驾驶员编号

# [人工审核] 布尔索引筛选：线路号 >= 1101 且 <= 1120（共20条线路）
filtered_df = df[(df[route_col] >= 1101) & (df[route_col] <= 1120)].copy()
print(f"\n>> 筛选线路 1101~1120, 共 {len(filtered_df)} 条记录")

# --- 5.2 创建输出目录 ---
# [人工审核] 按作业要求在根目录下创建名为 "线路驾驶员信息" 的文件夹
output_dir = '线路驾驶员信息'
os.makedirs(output_dir, exist_ok=True)  # exist_ok=True 避免重复创建报错

# --- 5.3 为每条线路生成 txt 文件 ---
unique_routes = sorted(filtered_df[route_col].unique())  # 按线路号升序排列
file_count = 0

for route in unique_routes:
    # [人工审核] 筛选属于当前线路的所有记录
    route_data = filtered_df[filtered_df[route_col] == route]
    # [人工审核] drop_duplicates() 去除重复的 (车辆编号, 驾驶员编号) 组合
    # [人工审核] 然后按车辆编号升序排列
    pairs = route_data[[vehicle_col, driver_col]].drop_duplicates()
    pairs = pairs.sort_values(vehicle_col)

    filepath = os.path.join(output_dir, f'{route}.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(f'线路号: {route}\n')              # 首行：线路号
        for _, row in pairs.iterrows():
            f.write(f'{int(row[vehicle_col])}\t{int(row[driver_col])}\n')  # 车辆编号\t驾驶员编号
    file_count += 1

print(f"\n>> 已在 '{output_dir}/' 目录下生成 {file_count} 个 txt 文件")
print(f"   线路列表: {unique_routes}")

# [人工审核] 打印第一个文件内容作为验证样例
sample_file = os.path.join(output_dir, f'{unique_routes[0]}.txt')
print(f"\n>> 样例文件 '{unique_routes[0]}.txt' 内容:")
with open(sample_file, 'r', encoding='utf-8') as f:
    print(f.read())


【任务5】线路驾驶员信息提取

>> 筛选线路 1101~1120, 共 35100 条记录

>> 已在 '线路驾驶员信息/' 目录下生成 20 个 txt 文件
   线路列表: [np.int64(1101), np.int64(1102), np.int64(1103), np.int64(1104), np.int64(1105), np.int64(1106), np.int64(1107), np.int64(1108), np.int64(1109), np.int64(1110), np.int64(1111), np.int64(1112), np.int64(1113), np.int64(1114), np.int64(1115), np.int64(1116), np.int64(1117), np.int64(1118), np.int64(1119), np.int64(1120)]

>> 样例文件 '1101.txt' 内容:
线路号: 1101
9127	1319
9128	1600
9129	1317
9130	41
9131	1873
9132	1599
9136	1609
9137	2293
9138	1593
9140	143
9141	147
9142	2150
9143	1130
9144	167
9145	171
9146	1997
9147	1996
9148	1659
9149	1127
9150	1316
9151	1661
9152	108
9153	1809
9154	192
9155	168
9156	131
9158	185
9160	2240
9161	1866
9163	182
9166	199
9167	1136
9168	202
9169	206
9171	1122
9172	208
9173	1594
9174	1827
9175	184
9176	146
9177	149
9179	2149
9180	2287
9181	100
9182	1999
9184	102
9185	1622
9188	127
9190	2292
9191	2532
9192	27
9193	2374
9194	2295
9195	1995
9196	1608
9198	1601
9200	2534
9204	1

In [7]:
# ============================================================
# 任务6: 运行效率热力图
# ============================================================
print("\n" + "=" * 60)
print("【任务6】运行效率热力图")
print("=" * 60)

# ── 6.1 统计各维度的服务人次（每行 = 1 人次）──
# [人工审核] 驾驶员维度：value_counts() 按降序统计每个驾驶员的服务人次
driver_rides = df[driver_col].value_counts().head(10)
top10_drivers = driver_rides.index.tolist()
print(f"\n>> Top 10 驾驶员 (人次):")
for i, (driver, count) in enumerate(driver_rides.items(), 1):
    print(f"   {i}. 驾驶员 {int(driver)}: {count} 次")

# [人工审核] 线路维度：统计每条线路的乘客人次
route_rides = df[route_col].value_counts().head(10)
top10_routes = route_rides.index.tolist()
print(f"\n>> Top 10 线路 (人次):")
for i, (route, count) in enumerate(route_rides.items(), 1):
    print(f"   {i}. 线路 {int(route)}: {count} 次")

# [人工审核] 上车站点维度：统计每个站点的上车人次
boarding_rides = df[boarding_col].value_counts().head(10)
top10_stations = boarding_rides.index.tolist()
print(f"\n>> Top 10 上车站点 (人次):")
for i, (station, count) in enumerate(boarding_rides.items(), 1):
    print(f"   {i}. 站点 {int(station)}: {count} 次")

# [人工审核] 车辆维度：统计每辆车的服务人次
vehicle_rides = df[vehicle_col].value_counts().head(10)
top10_vehicles = vehicle_rides.index.tolist()
print(f"\n>> Top 10 车辆 (人次):")
for i, (vehicle, count) in enumerate(vehicle_rides.items(), 1):
    print(f"   {i}. 车辆 {int(vehicle)}: {count} 次")

# ── 6.2 构建 4×10 矩阵用于热力图 ──
# [人工审核] 4 行 = 4 个分析维度（驾驶员/线路/上车站点/车辆）
# [人工审核] 10 列 = 每个维度的 Top 1 ~ Top 10
# [人工审核] 矩阵的每个元素值 = 该维度该排名的服务人次
heatmap_data = np.zeros((4, 10), dtype=int)

# [人工审核] 行0 = 驾驶员人次 (Top1 ~ Top10)
heatmap_data[0, :] = driver_rides.values
# 行1 = 线路人次 (Top1 ~ Top10)
heatmap_data[1, :] = route_rides.values
# 行2 = 上车站点人次 (Top1 ~ Top10)
heatmap_data[2, :] = boarding_rides.values
# 行3 = 车辆人次 (Top1 ~ Top10)
heatmap_data[3, :] = vehicle_rides.values

# ── 6.3 seaborn heatmap 绘制 ──
fig, ax = plt.subplots(figsize=(14, 6))

# [人工审核] 行标签和列标签分别对应 4 个维度和 Top 1~10
row_labels = ['Driver', 'Route', 'Boarding Station', 'Vehicle']
col_labels = [f'Top{i}' for i in range(1, 11)]

# [人工审核] seaborn.heatmap：annot=True 在每格标注数值，fmt='d' 整数显示
# [人工审核] cmap='RdYlGn'：红-黄-绿渐变，高值偏绿、低值偏红
# [人工审核] linewidths=0.5：格间白线宽度，增强可读性
sns.heatmap(
    heatmap_data,
    annot=True,                    # 每格显示数值
    fmt='d',                        # 整数格式
    cmap='RdYlGn',                  # 红-黄-绿配色
    xticklabels=col_labels,         # X轴：Top1~Top10
    yticklabels=row_labels,         # Y轴：四个维度
    linewidths=0.5,                 # 格间白色分隔线
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Number of Rides'}  # 颜色条标签
)

# [人工审核] 图表标题和轴标签使用英文（作业要求）
ax.set_title('Service Performance Heatmap: Top 10 Entities by Ride Count',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Rank', fontsize=12)
ax.set_ylabel('Dimension', fontsize=12)

# [人工审核] x 轴标签不旋转（作业要求 rotation=0）
plt.setp(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('performance_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"\n>> 6 热力图已保存为 performance_heatmap.png (dpi=150, bbox_inches='tight')")

# ── 6.4 分析说明（约50字，作业要求不超过50词）──
print("\n>> 热力图分析说明:")
analysis_text = (
    "From the heatmap, Route 46003 ranks first with 7127 rides, significantly "
    "outperforming the second-ranked route 1091 (4893 rides). Driver 0 and Vehicle 0 "
    "show anomalously high volumes (6484 and 11774 respectively), which may indicate "
    "placeholder or aggregated records rather than individual entities. Boarding "
    "Station 1 leads with 7776 rides, suggesting it is a major transit hub. The "
    "ride counts exhibit a steep drop-off from Top1 to Top10 across all four "
    "dimensions, revealing highly concentrated service demand on a small number "
    "of routes, stations, drivers, and vehicles."
)
print(f"   {analysis_text}")

print("\n" + "=" * 60)
print("全部任务完成！")
print("=" * 60)
print("\n生成的文件列表:")
print("  - ICData.csv")
print("  - homework.py (本文件)")
print("  - hour_distribution.png")
print("  - route_stops.png")
print("  - performance_heatmap.png")
print("  - 线路驾驶员信息/ (20个txt文件)")
print("  - README.md (人机协同报告)")


【任务6】运行效率热力图

>> Top 10 驾驶员 (人次):
   1. 驾驶员 0: 6484 次
   2. 驾驶员 90422201: 4801 次
   3. 驾驶员 1090101: 1334 次
   4. 驾驶员 90417292: 1107 次
   5. 驾驶员 90805213: 395 次
   6. 驾驶员 101: 380 次
   7. 驾驶员 90839201: 369 次
   8. 驾驶员 9017887: 358 次
   9. 驾驶员 9011030: 345 次
   10. 驾驶员 9011104: 329 次

>> Top 10 线路 (人次):
   1. 线路 46003: 7127 次
   2. 线路 1091: 4893 次
   3. 线路 9: 4474 次
   4. 线路 88: 4350 次
   5. 线路 5: 4190 次
   6. 线路 1101: 3782 次
   7. 线路 56012: 3736 次
   8. 线路 1112: 3683 次
   9. 线路 1114: 3414 次
   10. 线路 1115: 3244 次

>> Top 10 上车站点 (人次):
   1. 站点 1: 7776 次
   2. 站点 14: 6553 次
   3. 站点 17: 6449 次
   4. 站点 13: 5990 次
   5. 站点 8: 5894 次
   6. 站点 2: 5892 次
   7. 站点 9: 5813 次
   8. 站点 10: 5760 次
   9. 站点 15: 5736 次
   10. 站点 21: 5707 次

>> Top 10 车辆 (人次):
   1. 车辆 0: 11774 次
   2. 车辆 1000000: 2033 次
   3. 车辆 25263: 604 次
   4. 车辆 25321: 518 次
   5. 车辆 25282: 482 次
   6. 车辆 19683: 444 次
   7. 车辆 17468: 405 次
   8. 车辆 24915: 403 次
   9. 车辆 15506: 402 次
   10. 车辆 19881: 399 次



>> 6 热力图已保存为 performance_heatmap.png (dpi=150, bbox_inches='tight')

>> 热力图分析说明:
   From the heatmap, Route 46003 ranks first with 7127 rides, significantly outperforming the second-ranked route 1091 (4893 rides). Driver 0 and Vehicle 0 show anomalously high volumes (6484 and 11774 respectively), which may indicate placeholder or aggregated records rather than individual entities. Boarding Station 1 leads with 7776 rides, suggesting it is a major transit hub. The ride counts exhibit a steep drop-off from Top1 to Top10 across all four dimensions, revealing highly concentrated service demand on a small number of routes, stations, drivers, and vehicles.

全部任务完成！

生成的文件列表:
  - ICData.csv
  - homework.py (本文件)
  - hour_distribution.png
  - route_stops.png
  - performance_heatmap.png
  - 线路驾驶员信息/ (20个txt文件)
  - README.md (人机协同报告)
